In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, BatchNormalization, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# function to load and preprocess audio data by extracting features (MFCCs)
def load_and_preprocess_data(audio_path, label):
    # Load the audio file and set sample rate to None to keep it original
    audio, sr = librosa.load(audio_path, sr=None)

    # extract MFCC (Mel-frequency cepstral coefficient) features
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
  
    # Transpose MFCCs to shape them correctly for model input       
    return mfccs.T, label

# Function to prepare dataset from real and fake audio file paths
def prepare_dataset(real_paths, fake_paths):
    x, y = [], []  # x will hold features and y will hold labels (0 for real, 1 for fake)

    # load real audio files and label them as 0 (real)
    for path in real_paths:
        features, label = load_and_preprocess_data(path, 0)
        x.append(features)
        y.append(label)

    # load fake audio files and label them as 1 (fake)
    for path in fake_paths:
        features, label = load_and_preprocess_data(path, 1)
        x.append(features)
        y.append(label)

    return x, np.array(y)


                
        

In [ ]:
def augment_data(x, y):
    augmented_x, augmented_y = [], []

    # Loop through each feature set and label
    for features, label in zip(x, y):
        augmented_x.append(features)
        augmented_y.append(label)

        # Time Stretching
        audio_signal = librosa.effects.time_stretch(features.T[0], rate=0.8)
        augmented_x.append(librosa.feature.mfcc(y=audio_signal, sr=22050, n_mfcc=13).T)
        augmented_y.append(label)

        # Pitch shifting
        audio_signal_shifted = librosa.effects.pitch_shift(features.T[0], sr=22050, n_steps=2)
        augmented_x.append(librosa.feature.mfcc(y=audio_signal_shifted, sr=22050, n_mfcc=13).T)
        augmented_y.append(label)

    # return augmented features and labels
    return augmented_x, np.array(augmented_y)

In [ ]:


def create_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        Masking(mask_value=0),

        LSTM(64, return_sequences=True),
        Dropout(0.3),

        LSTM(32),
        Dropout(0.3),

        Dense(16, activation='relu'),
        Dropout(0.2),

        Dense(2, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),  # ✅ only once
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
# Function to train and evaluate the model
def train_and_evaluate(model, X_train, Y_train, X_test, Y_test, epochs=20, batch_size=32):


    early_stop= EarlyStopping(
        monitor='val_loss',
        patience=3,
    restore_best_weights=True)
    
    # Train the model
    history = model.fit(
        X_train, Y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    # Evaluate the model
    test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
    print(f"Test accuracy: {test_acc:.4f}")

    # Return training history
    return history

In [ ]:
# Function to visualize training and validation accuracy and loss
def plot_training_history(history):
    plt.figure(figsize=(12, 4))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()
               

In [ ]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true.argmax(axis=1), y_pred.argmax(axis=1))
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

In [ ]:
import os

In [ ]:
base_path = "D:/AISTRIPPER/Graphic-Era_2025-26-AI_STRIPPER/dataset/KAGGLE/training"

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from tensorflow.keras.layers import Input, Masking

In [ ]:
# Get file paths
real_paths = [os.path.join(base_path, "REAL", f)
              for f in os.listdir(os.path.join(base_path, "REAL"))
              if f.endswith('.wav')]

fake_paths = [os.path.join(base_path, "FAKE", f)
              for f in os.listdir(os.path.join(base_path, "FAKE"))
              if f.endswith('.wav')]


# Prepare and augment the dataset
X, y = prepare_dataset(real_paths, fake_paths)
X_augmented, y_augmented = augment_data(X, y)


# Pad sequences
max_length = max(len(seq) for seq in X_augmented)
X_padded = pad_sequences(X_augmented, maxlen=max_length, dtype='float32',
                         padding='post', truncating='post')


# Split the data (only once)
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_augmented, test_size=0.2, random_state=42
)


# Convert labels to categorical
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)


# Create and train the model
model = create_model(input_shape=(X_train.shape[1], X_train.shape[2]))
history = train_and_evaluate(model, X_train, y_train, X_test, y_test)


# Visualization
plot_training_history(history)

y_pred = model.predict(X_test)
plot_confusion_matrix(y_test, y_pred)

print(classification_report(
    y_test.argmax(axis=1),
    y_pred.argmax(axis=1)
))


# Optional: Print dataset statistics
print(f"Number of real audio samples: {len(real_paths)}")
print(f"Number of fake audio samples: {len(fake_paths)}")
print(f"Total Number of samples: {len(real_paths) + len(fake_paths)}")
print(f"Shape of padded dataset: {X_padded.shape}")
print(f"Maximum sequence length: {max_length}")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")


In [ ]:
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def preprocess_audio(audio_path, max_length):
    # Load the audio
    audio, sr = librosa.load(audio_path, sr=None)

    # Extract MFCC features 
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    # Transpose the MFCCs to match the input format
    mfccs = mfccs.T

    # Pad the sequence to the max length used during training
    padded_mfccs = pad_sequences(
        [mfccs],
        X_augmented,
        maxlen=max_length,
        dtype='float32',
        padding='post',
        truncating='post'
    )

    return padded_mfccs


In [ ]:
test_audio_paths = [
    # '/kaggle/input/deep-voice-deepfake-voice-recognition/KAGGLE/AUDIO/FAKE/trump-to-taylor.wav',
    # '/kaggle/input/deep-voice-deepfake-voice-recognition/KAGGLE/AUDIO/REAL/trump-to-taylor.wav'
    r'D:\AISTRIPPER\Graphic-Era_2025-26-AI_STRIPPER\dataset\KAGGLE\testing\FAKE\file2.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav',
    r'D:\AISTRIPPER\Graphic-Era_2025-26-AI_STRIPPER\dataset\KAGGLE\testing\REAL\file6.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav'
]

# for audio_path in test_audio_paths:
#     padded_sample = preprocess_audio(audio_path, max_length=X_train.shape[1])
    
#     prediction = model.predict(padded_sample)
#     predicted_class = np.argmax(prediction, axis=1)[0]

#     if predicted_class == 1:
#         print(f"{audio_path}: Fake")
#     else:
#         print(f"{audio_path}: Real")

for audio_path in test_audio_paths:
    padded_sample = preprocess_audio(audio_path, max_length=X_train.shape[1])
    
    prediction = model.predict(padded_sample)
    predicted_class = np.argmax(prediction, axis=1)[0]

    if predicted_class == 1:
        print(f"{audio_path}: Fake")
    else:
        print(f"{audio_path}: Real")

In [ ]:
model.save('lstm_model.h5')

In [ ]:
from tensorflow.keras.models import load_model
newmodel = load_model('/kaggle/working/lstm_model.h5')

In [ ]:
prediction = newmodel.predict(padded_sample)
predicted_class = np.argmax(prediction, axis=1)[0]

if predicted_class == 1:
    print(f"{audio_path}: Fake")
else:
    print(f"{audio_path}: Real")